## Structured Output 
### model can be requested to provide their response in a format, matching a given schema. This is useful for ensuring the output can be easily passed and used in subsequent processing. Lang chain supports multiple schema types, and methods for enforcing structured output.

### Pydantic
Pedantic models provide the richest feature set with field validation, description, and nested structures

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model= init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1084ea8d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10d83a710>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description=" This is the year the movie was released")
    director: str=Field(description=" The director of the movie")
    rating:float= Field(description="The movie rating out of 5")


In [3]:
model_with_structure= model.with_structured_output(Movie)

In [4]:
model_with_structure.invoke(" provide details about the movie Avengers End game")

Movie(title='Avengers: Endgame', year=2019, director='Anthony Russo, Joseph Russo', rating=4.5)

In [7]:
### this is one of the way you can handle the out of context question

import os
from pydantic import BaseModel, Field
from typing import Optional
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

# Ensure your API key is set
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# 1. Initialize the Groq model dynamically using init_chat_model
# Notice the prefix "groq:" before the model name
model = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0)

# 2. Define the core entity schema
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This is the year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 5")

# 3. Define the wrapper schema to handle out-of-context questions
class MovieQueryResponse(BaseModel):
    is_movie_query: bool = Field(
        description="True if the user's prompt is about a movie or asks for movie details. False otherwise."
    )
    movie_info: Optional[Movie] = Field(
        default=None,
        description="The extracted movie details. ONLY populate this if is_movie_query is True."
    )
    fallback_message: Optional[str] = Field(
        default=None,
        description="A polite message explaining that you only provide information about movies. ONLY populate this if is_movie_query is False."
    )

# 4. Bind the schema to the model
model_with_structure = model.with_structured_output(MovieQueryResponse)

# 5. Create a prompt template to guide the model's behavior
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized movie assistant. 
    Your ONLY job is to extract or provide details about movies based on the user's input.
    
    If the user asks about ANY topic other than movies, 
    you must set 'is_movie_query' to False and provide a polite 'fallback_message'.
    Do not attempt to answer non-movie questions."""),
    ("user", "{user_input}")
])

# 6. Create the chain
chain = prompt | model_with_structure

# 7. Test the chain
response = chain.invoke({"user_input": "provide details about KGF chapter 1"})

# 8. Handle the output safely
if response.is_movie_query and response.movie_info:
    print(f"Title: {response.movie_info.title}")
    print(f"Year: {response.movie_info.year}")
    print(f"Director: {response.movie_info.director}")
    print(f"Rating: {response.movie_info.rating}")
else:
    print(f"Fallback: {response.fallback_message}")

Title: KGF Chapter 1
Year: 2018
Director: Prashanth Neel
Rating: 4.5


### NEsted structure


In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    Name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

response= model_with_structure.invoke(" details about movie KGF chapter 1")
print(response)

title='KGF Chapter 1' year=2018 cast=[Actor(Name='Yash', role='Rocky'), Actor(Name='Srinidhi Shetty', role='Reena')] genres=['Action', 'Adventure', 'Drama'] budget=80.0


## TypeDict
### TypedDict provide a simpler alternative using python's built-in typing, ideal when you don't need runtime validation.

In [12]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """ A movie with details"""
    title: Annotated[str, ..., "The title of the movie "]
    year: Annotated[int,...," The Year of movie release"]
    director: Annotated[str,...," the year the movie was released"]
    rating: Annotated[float,...,"Rating of the movie out of 10"]


In [13]:
typeddist_model=model.with_structured_output(MovieDict)

In [14]:
typeddist_model.invoke(" I need details of movie The pursuit of Happiness")

{'director': 'Gabriele Muccino',
 'rating': 7.9,
 'title': 'The Pursuit of Happyness',
 'year': 2006}

In [15]:

class Actor(TypedDict):
    Name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

response= model_with_structure.invoke(" details about movie KGF chapter 1")
print(response)

{'budget': 80, 'cast': [{'Name': 'Yash', 'role': 'Rocky'}, {'Name': 'Srinidhi Shetty', 'role': 'Reena'}], 'genres': ['Action', 'Adventure', 'Drama'], 'title': 'KGF Chapter 1', 'year': 2018}


In [16]:
model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## Data Classes
### A data class is a class typically containing mainly data, although there aren't really any restrictions .You created using the @dataclass decorator

In [17]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact information for a reason"""
    name: str= Field(description="the name of the person")
    email: str=Field(description=" they email id of the person")
    phone: str=Field(description="phone number of the person")

agent = create_agent(
    model='gpt-4o-mini',  # Fixed: Use a valid OpenAI model
    response_format=ContactInfo
)

result = agent.invoke(
    {
        # Fixed: plural "messages"
        "messages": [{"role": "user", "content": "Extract the contact info from Uday Venkatesha, udayvenkatesh2015@gmail.com, 8729045044"}]
    }
)

print(result['structured_response'])


name='Uday Venkatesha' email='udayvenkatesh2015@gmail.com' phone='8729045044'


In [20]:
##dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person"""
    name: str # the name of the person
    email: str
    phone: str

agent = create_agent(
    model='gpt-4o-mini',  # Fixed: Use a valid OpenAI model
    response_format=ContactInfo
)

result = agent.invoke(
    {
        # Fixed: plural "messages"
        "messages": [{"role": "user", "content": "Extract the contact info from Uday Venkatesha, udayvenkatesh2015@gmail.com, 8729045044"}]
    }
)

print(result['structured_response'])
    

ContactInfo(name='Uday Venkatesha', email='udayvenkatesh2015@gmail.com', phone='8729045044')
